# Amazon ML Challenge 2026 — FAST Colab Submission Pipeline

This notebook is the **deadline-oriented full test inference pipeline** for the current entity-resolution model checkpoint.

Important: this version uses **TOP-3 candidates per S1** to reduce runtime. That is a speed fallback, not the previously validated K100/K150 strategy.

Fixes included: DuckDB temp-directory retry handling, DuckDB alias issue, NumPy-array/list feature handling, memory-safe S1 batching, removal of stale `s1_all` references, and per-batch garbage collection.


## FAST RUN SETTINGS

```text
Model features:        35
Candidate K:            3
Decision threshold:    0.65
S1 batch size:       10,000
Resume:                True
Smoke test:           False (the K3 path was already smoke-tested)
```

The full S1 table is **never loaded into RAM at once**. Only one S1 batch is queried from DuckDB at a time.


In [ ]:
# ============================================================
# 1. COLAB INSTALLS
# ============================================================
%pip install -q duckdb pyarrow rapidfuzz Unidecode catboost joblib psutil

In [ ]:
# ============================================================
# 2. IMPORTS + COLAB BOOTSTRAP
# ============================================================
import os
import re
import json
import math
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import joblib
import duckdb
from rapidfuzz import fuzz
from unidecode import unidecode
from catboost import CatBoostClassifier

print("Imports loaded.")


In [ ]:
# ============================================================
# 3. MOUNT GOOGLE DRIVE + CONFIGURATION
# ============================================================
from google.colab import drive

drive.mount("/content/drive")

MY_DRIVE = Path("/content/drive/MyDrive")
RESOURCE_DIR = MY_DRIVE / "student_resource"
ARTIFACT_DIR = MY_DRIVE / "amazon_ml_10k_artifacts"

TEST_DIR = RESOURCE_DIR / "dataset" / "test"
TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

# ------------------------------------------------------------
# Run controls — FAST DEADLINE MODE
# ------------------------------------------------------------
SMOKE_TEST = False
SMOKE_TEST_S1 = 2000

RESUME = True
OVERWRITE_CLEAN = False
OVERWRITE_DB = False

# Larger batch reduces repeated DuckDB/Pandas overhead while
# remaining memory-safe because only one S1 batch is loaded.
BLOCK_BATCH_S1 = 10_000

PREPROCESS_CHUNK = 100_000
TFIDF_PAIR_BATCH = 50_000

# Emergency deadline mode: retrieve and score only the top 3.
BASE_K = 3
EXPANSION_K = 3
DECISION_THRESHOLD = 0.65

SEED = 42

# Validated blocking configuration.
MAX_NAME_POSTING = 150
MAX_ADDR_POSTING = 200
MAX_QGRAM_POSTING = 250
MAX_PREFIX_POSTING = 300

RARE_NAME_KEYS = 2
RARE_ADDR_KEYS = 2
RARE_QGRAM_KEYS = 3

MAX_COMPOSITE_POSTING = 100
MAX_ADDR_TOKENS_FOR_PAIRS = 6
MAX_ADDR_NUMS_FOR_PAIRS = 4

BLOCK_WEIGHTS = {
    "exact_native_name": 10.0,
    "exact_ascii_name": 9.0,
    "postal": 6.0,
    "rare_name_native": 4.0,
    "rare_name_ascii": 4.0,
    "rare_addr_native": 2.5,
    "rare_addr_ascii": 2.5,
    "qgram": 1.0,
    "prefix4": 0.5,
}

TARGET_WEIGHTS = {
    "squashed_exact": 8.0,
    "squashed_qgram": 1.25,
    "addr_num": 3.5,
    "addr_pair": 3.0,
}

DUCKDB_THREADS = max(2, min(8, (os.cpu_count() or 4)))
available_gb = psutil.virtual_memory().available / (1024 ** 3)
DUCKDB_MEMORY_GB = max(4, int(available_gb * 0.55))
DUCKDB_MEMORY = f"{DUCKDB_MEMORY_GB}GB"

SUBMISSION_ROOT = MY_DRIVE / "amazon_ml_final_submission"
OUTPUT_DIR = SUBMISSION_ROOT / ("smoke_output" if SMOKE_TEST else "output_fast_k3")

# Heavy temporary work stays on Colab local disk for speed.
WORK_DIR = Path("/content/amazon_ml_final_work")
CLEAN_DIR = WORK_DIR / "clean"
INDEX_DIR = WORK_DIR / "duckdb"
RUN_DIR = WORK_DIR / ("smoke_run_fast_k3" if SMOKE_TEST else "full_run_fast_k3")
TMP_DIR = WORK_DIR / "tmp"

for p in [WORK_DIR, CLEAN_DIR, INDEX_DIR, RUN_DIR, TMP_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    BLOCK_BATCH_S1 = min(BLOCK_BATCH_S1, 500)

print("MY_DRIVE         :", MY_DRIVE)
print("RESOURCE_DIR     :", RESOURCE_DIR)
print("ARTIFACT_DIR     :", ARTIFACT_DIR)
print("TEST_DIR         :", TEST_DIR)
print("WORK_DIR         :", WORK_DIR)
print("RUN_DIR          :", RUN_DIR)
print("OUTPUT_DIR       :", OUTPUT_DIR)
print("DUCKDB_MEMORY    :", DUCKDB_MEMORY)
print("DUCKDB_THREADS   :", DUCKDB_THREADS)
print("SMOKE_TEST       :", SMOKE_TEST)
print("BASE_K           :", BASE_K)
print("EXPANSION_K      :", EXPANSION_K)
print("BLOCK_BATCH_S1   :", BLOCK_BATCH_S1)


In [ ]:
# ============================================================
# 4. VERIFY INPUTS + LOAD SAVED ARTIFACTS
# ============================================================
required_inputs = [TEST_S1, TEST_S2, TEST_S3]
required_artifacts = [
    ARTIFACT_DIR / "checkpoint_model_8k.cbm",
    ARTIFACT_DIR / "name_char_tfidf.joblib",
    ARTIFACT_DIR / "name_word_tfidf.joblib",
    ARTIFACT_DIR / "address_char_tfidf.joblib",
    ARTIFACT_DIR / "address_word_tfidf.joblib",
    ARTIFACT_DIR / "feature_columns.json",
    ARTIFACT_DIR / "model_config.json",
]

for p in required_inputs + required_artifacts:
    print(("FOUND " if p.exists() else "MISSING"), p)

missing = [str(p) for p in required_inputs + required_artifacts if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

with open(ARTIFACT_DIR / "feature_columns.json", "r", encoding="utf-8") as f:
    FEATURE_COLUMNS = json.load(f)

with open(ARTIFACT_DIR / "model_config.json", "r", encoding="utf-8") as f:
    SAVED_CONFIG = json.load(f)

MODEL = CatBoostClassifier()
MODEL.load_model(str(ARTIFACT_DIR / "checkpoint_model_8k.cbm"))

NAME_CHAR_VECTORIZER = joblib.load(ARTIFACT_DIR / "name_char_tfidf.joblib")
NAME_WORD_VECTORIZER = joblib.load(ARTIFACT_DIR / "name_word_tfidf.joblib")
ADDRESS_CHAR_VECTORIZER = joblib.load(ARTIFACT_DIR / "address_char_tfidf.joblib")
ADDRESS_WORD_VECTORIZER = joblib.load(ARTIFACT_DIR / "address_word_tfidf.joblib")

print("\nLoaded feature count:", len(FEATURE_COLUMNS))
print("Saved candidate config:", SAVED_CONFIG.get("candidate_strategy"))
print("Saved threshold       :", SAVED_CONFIG.get("current_threshold"))
print("Model feature names   :", len(getattr(MODEL, "feature_names_", [])))

assert len(FEATURE_COLUMNS) == 35, FEATURE_COLUMNS
if getattr(MODEL, "feature_names_", None):
    assert list(MODEL.feature_names_) == FEATURE_COLUMNS, (
        "Model feature names do not exactly match feature_columns.json"
    )

In [ ]:
# ============================================================
# 5. EXACT NORMALIZATION / FEATURE HELPERS
# ============================================================
_ws_re = re.compile(r"\s+")
_nonword_re = re.compile(r"[^\w\s]+", flags=re.UNICODE)
_token_re = re.compile(r"\w+", flags=re.UNICODE)
_ascii_alnum_re = re.compile(r"[^a-z0-9]+")

GENERIC_NAME_TOKENS = {
    "the", "and", "co", "company", "corp", "corporation", "inc", "incorporated",
    "llc", "ltd", "limited", "plc", "pvt", "private", "enterprises", "enterprise",
    "group", "holdings", "services", "service", "solutions", "international",
    "india", "ind", "usa", "us", "store", "shop", "mart", "market", "business",
    "trading", "traders", "works", "industries", "industry", "center", "centre"
}

GENERIC_ADDR_TOKENS = {
    "road", "rd", "street", "st", "lane", "ln", "avenue", "ave", "sector",
    "near", "opposite", "opp", "behind", "beside", "building", "bldg",
    "floor", "fl", "shop", "plot", "house", "home", "block", "district",
    "city", "state", "county", "india", "usa", "us"
}

def safe_text(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except (TypeError, ValueError):
        pass
    return str(x)

def norm_native(x):
    x = safe_text(x).lower().strip()
    if not x:
        return ""
    x = _nonword_re.sub(" ", x)
    return _ws_re.sub(" ", x).strip()

def norm_ascii(x):
    x = safe_text(x)
    if not x:
        return ""
    x = unidecode(x).lower()
    x = _ascii_alnum_re.sub(" ", x)
    return _ws_re.sub(" ", x).strip()

def token_list(text, generic):
    text = safe_text(text)
    if not text:
        return []
    out = []
    seen = set()
    for t in _token_re.findall(text):
        t = t.lower()
        if len(t) < 3 or t in generic or t in seen:
            continue
        seen.add(t)
        out.append(t)
    return out

def postal_codes(text):
    text = safe_text(text)
    return list(dict.fromkeys(re.findall(r"(?<!\d)(\d{5,6})(?!\d)", text)))

def numeric_tokens(text):
    text = safe_text(text)
    return list(dict.fromkeys(re.findall(r"(?<!\w)\d{2,}(?!\w)", text)))

def char_ngrams(text, n=3):
    text = safe_text(text)
    if len(text) < n:
        return []
    x = re.sub(r"[^a-z0-9]", "", text)
    if len(x) < n:
        return []
    return list(dict.fromkeys(x[i:i+n] for i in range(len(x)-n+1)))

def prefix_key(text, n=4):
    x = re.sub(r"[^a-z0-9]", "", safe_text(text))
    return x[:n] if len(x) >= n else ""

def squash_ascii(text):
    return re.sub(r"[^a-z0-9]", "", safe_text(text))

def prepare_chunk(df, rid_col, rid_start, source=None):
    required = ["entity_id", "business_name", "business_address", "country"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    out = pd.DataFrame(index=df.index)
    out[rid_col] = np.arange(rid_start, rid_start + len(df), dtype=np.int64)
    out["entity_id"] = df["entity_id"].map(safe_text)
    out["country"] = df["country"].map(lambda x: safe_text(x).lower())

    raw_name = df["business_name"].map(safe_text)
    raw_addr = df["business_address"].map(safe_text)

    out["name_native"] = raw_name.map(norm_native)
    out["name_ascii"] = raw_name.map(norm_ascii)
    out["addr_native"] = raw_addr.map(norm_native)
    out["addr_ascii"] = raw_addr.map(norm_ascii)

    out["name_tokens_native"] = out["name_native"].map(
        lambda x: token_list(x, GENERIC_NAME_TOKENS)
    )
    out["name_tokens_ascii"] = out["name_ascii"].map(
        lambda x: token_list(x, GENERIC_NAME_TOKENS)
    )
    out["addr_tokens_native"] = out["addr_native"].map(
        lambda x: token_list(x, GENERIC_ADDR_TOKENS)
    )
    out["addr_tokens_ascii"] = out["addr_ascii"].map(
        lambda x: token_list(x, GENERIC_ADDR_TOKENS)
    )

    out["postal_native"] = raw_addr.map(postal_codes)
    out["numbers_native"] = out["addr_native"].map(numeric_tokens)
    out["name_prefix4"] = out["name_ascii"].map(prefix_key)
    out["name_qgrams"] = out["name_ascii"].map(lambda x: char_ngrams(x, 3))
    out["name_squashed"] = out["name_ascii"].map(squash_ascii)

    if source is not None:
        out["source"] = source

    return out.reset_index(drop=True)

def _to_list(x):
    """Normalize Python/NumPy/Pandas list-like values to a Python list."""
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, pd.Series):
        return x.tolist()
    if isinstance(x, (list, tuple, set)):
        return list(x)
    try:
        if pd.isna(x):
            return []
    except (TypeError, ValueError):
        pass
    return [x]

def jaccard_lists(a, b):
    sa = set(_to_list(a))
    sb = set(_to_list(b))
    u = sa | sb
    return float(len(sa & sb) / len(u)) if u else 0.0

def ratio_series(a, b, fn=fuzz.ratio):
    return np.fromiter(
        (fn(x, y) / 100.0 if x and y else 0.0 for x, y in zip(a, b)),
        dtype=np.float32,
        count=len(a),
    )

def list_jaccard_series(a, b):
    return np.fromiter(
        (jaccard_lists(x, y) for x, y in zip(a, b)),
        dtype=np.float32,
        count=len(a),
    )

def shared_count_series(a, b):
    return np.fromiter(
        (len(set(_to_list(x)) & set(_to_list(y))) for x, y in zip(a, b)),
        dtype=np.int16,
        count=len(a),
    )

def number_overlap_series(a, b):
    def one(x, y):
        sx, sy = set(_to_list(x)), set(_to_list(y))
        u = sx | sy
        return float(len(sx & sy) / len(u)) if u else 0.0
    return np.fromiter((one(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))

def number_shared_series(a, b):
    return np.fromiter(
        (len(set(_to_list(x)) & set(_to_list(y))) for x, y in zip(a, b)),
        dtype=np.int16,
        count=len(a),
    )

def len_ratio_series(a, b):
    return np.fromiter(
        (min(len(x), len(y)) / max(len(x), len(y)) if x and y else 0.0 for x, y in zip(a, b)),
        dtype=np.float32,
        count=len(a),
    )

print("Normalization and feature helpers loaded.")

In [ ]:
# ============================================================
# 6. STREAMING PREPROCESSOR -> PARQUET
# ============================================================
def count_parquet_rows(directory, rid_col):
    files = sorted(Path(directory).glob("part-*.parquet"))
    if not files:
        return 0
    total = 0
    for f in files:
        total += len(pd.read_parquet(f, columns=[rid_col]))
    return total

def preprocess_source_to_parquet(
    input_path,
    output_dir,
    rid_col,
    rid_start=0,
    source=None,
    overwrite=False,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if overwrite:
        for f in output_dir.glob("part-*.parquet"):
            f.unlink()

    existing = sorted(output_dir.glob("part-*.parquet"))
    existing_parts = len(existing)
    existing_rows = count_parquet_rows(output_dir, rid_col)

    if existing_parts:
        print(f"Resuming {input_path.name}: {existing_rows:,} rows already written.")

    reader = pd.read_csv(
        input_path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=PREPROCESS_CHUNK,
        low_memory=False,
    )

    rows_written = existing_rows
    part_no = existing_parts
    t0 = time.time()

    for chunk_no, chunk in enumerate(reader):
        if chunk_no < existing_parts:
            continue

        out = prepare_chunk(
            chunk,
            rid_col=rid_col,
            rid_start=rid_start + rows_written,
            source=source,
        )
        out.to_parquet(output_dir / f"part-{part_no:05d}.parquet", index=False)

        rows_written += len(out)
        part_no += 1

        if part_no == existing_parts + 1 or part_no % 5 == 0:
            print(
                f"{input_path.name}: {rows_written:,} rows | "
                f"elapsed={time.time()-t0:,.1f}s"
            )

    print(
        f"Finished {input_path.name}: {rows_written:,} rows | "
        f"elapsed={time.time()-t0:,.1f}s"
    )
    return rows_written

In [ ]:
# ============================================================
# 7. PREPROCESS FULL TEST DATA
# ============================================================
TEST_S1_CLEAN = CLEAN_DIR / "s1"
TEST_S2_CLEAN = CLEAN_DIR / "s2"
TEST_S3_CLEAN = CLEAN_DIR / "s3"

s1_rows = preprocess_source_to_parquet(
    TEST_S1,
    TEST_S1_CLEAN,
    rid_col="s1_rid",
    rid_start=0,
    source="S1",
    overwrite=OVERWRITE_CLEAN,
)

s2_rows = preprocess_source_to_parquet(
    TEST_S2,
    TEST_S2_CLEAN,
    rid_col="s23_rid",
    rid_start=0,
    source="S2",
    overwrite=OVERWRITE_CLEAN,
)

s3_rows = preprocess_source_to_parquet(
    TEST_S3,
    TEST_S3_CLEAN,
    rid_col="s23_rid",
    rid_start=s2_rows,
    source="S3",
    overwrite=OVERWRITE_CLEAN,
)

print("\nTEST counts")
print("S1 :", f"{s1_rows:,}")
print("S2 :", f"{s2_rows:,}")
print("S3 :", f"{s3_rows:,}")
print("S23:", f"{s2_rows+s3_rows:,}")

## Full-scale index design

The next cells build a DuckDB database from the cleaned Parquet files. The database contains:

- the normalized S1/S23 records;
- exact-name frequency tables;
- native/ASCII token frequency tables;
- q-gram frequency tables;
- filtered posting tables under the blocker caps;
- postal/prefix postings;
- the squashed-name exact index;
- the address-number composite index;
- the address-token-pair composite index.

The index is built on the **candidate corpus only** (test S2 + S3), while S1 is used as the query side.

In [ ]:
# ============================================================
# 8. BUILD DISK-BACKED DUCKDB INDEX
# ============================================================
DB_PATH = INDEX_DIR / "final_test_block_index.duckdb"
DB_TMP = TMP_DIR / "duckdb"
DB_TMP.mkdir(parents=True, exist_ok=True)

def qpath(path):
    return str(Path(path)).replace("'", "''")

def connect_db(db_path, overwrite=False):
    """Open DuckDB with configuration applied at connection creation."""
    db_path = Path(db_path)

    old_con = globals().get("con")
    if old_con is not None:
        try:
            old_con.close()
        except Exception:
            pass

    if overwrite and db_path.exists():
        db_path.unlink()

    return duckdb.connect(
        str(db_path),
        config={
            "memory_limit": DUCKDB_MEMORY,
            "threads": str(DUCKDB_THREADS),
            "temp_directory": str(DB_TMP),
        },
    )

def table_exists(con, name):
    return bool(
        con.execute(
            "SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?",
            [name],
        ).fetchone()[0]
    )

def build_full_index(db_path, overwrite=False):
    con = connect_db(db_path, overwrite=overwrite)

    if (not overwrite and table_exists(con, "s23") and table_exists(con, "idx_addr_pair")):
        print("Existing completed DuckDB database detected; reusing it.")
        return con

    t0 = time.time()

    s1_glob = qpath(TEST_S1_CLEAN / "*.parquet")
    s2_glob = qpath(TEST_S2_CLEAN / "*.parquet")
    s3_glob = qpath(TEST_S3_CLEAN / "*.parquet")

    con.execute(f"CREATE OR REPLACE TABLE s1 AS SELECT * FROM read_parquet('{s1_glob}')")
    con.execute(f"""
        CREATE OR REPLACE TABLE s23 AS
        SELECT * FROM read_parquet('{s2_glob}')
        UNION ALL
        SELECT * FROM read_parquet('{s3_glob}')
    """)

    print("Loaded clean tables.")
    print("S1 :", con.execute("SELECT COUNT(*) FROM s1").fetchone()[0])
    print("S23:", con.execute("SELECT COUNT(*) FROM s23").fetchone()[0])

    # Exact-name / prefix frequencies.
    con.execute("""
        CREATE OR REPLACE TABLE name_native_freq AS
        SELECT name_native, COUNT(*)::BIGINT AS freq
        FROM s23 WHERE name_native <> ''
        GROUP BY name_native
    """)
    con.execute("""
        CREATE OR REPLACE TABLE name_ascii_freq AS
        SELECT name_ascii, COUNT(*)::BIGINT AS freq
        FROM s23 WHERE name_ascii <> ''
        GROUP BY name_ascii
    """)
    con.execute("""
        CREATE OR REPLACE TABLE prefix4_freq AS
        SELECT name_prefix4, COUNT(*)::BIGINT AS freq
        FROM s23 WHERE name_prefix4 <> ''
        GROUP BY name_prefix4
    """)

    # Token tables.
    con.execute("""
        CREATE OR REPLACE TABLE name_tokens AS
        SELECT DISTINCT s23_rid, 'native' AS kind, token
        FROM s23
        CROSS JOIN UNNEST(name_tokens_native) AS u(token)
        WHERE token <> ''
        UNION ALL
        SELECT DISTINCT s23_rid, 'ascii' AS kind, token
        FROM s23
        CROSS JOIN UNNEST(name_tokens_ascii) AS u(token)
        WHERE token <> ''
    """)

    con.execute("""
        CREATE OR REPLACE TABLE addr_tokens AS
        SELECT DISTINCT s23_rid, 'native' AS kind, token
        FROM s23
        CROSS JOIN UNNEST(addr_tokens_native) AS u(token)
        WHERE token <> ''
        UNION ALL
        SELECT DISTINCT s23_rid, 'ascii' AS kind, token
        FROM s23
        CROSS JOIN UNNEST(addr_tokens_ascii) AS u(token)
        WHERE token <> ''
    """)

    con.execute("""
        CREATE OR REPLACE TABLE name_token_freq AS
        SELECT kind, token, COUNT(*)::BIGINT AS freq
        FROM name_tokens
        GROUP BY kind, token
    """)
    con.execute("""
        CREATE OR REPLACE TABLE addr_token_freq AS
        SELECT kind, token, COUNT(*)::BIGINT AS freq
        FROM addr_tokens
        GROUP BY kind, token
    """)

    # Q-grams: only the frequencies and low-frequency postings are retained.
    con.execute("""
        CREATE OR REPLACE TABLE qgrams AS
        SELECT DISTINCT s23_rid, qgram
        FROM s23
        CROSS JOIN UNNEST(name_qgrams) AS u(qgram)
        WHERE qgram <> ''
    """)
    con.execute("""
        CREATE OR REPLACE TABLE qgram_freq AS
        SELECT qgram, COUNT(*)::BIGINT AS freq
        FROM qgrams
        GROUP BY qgram
    """)

    # Targeted squashed-name exact index.
    con.execute("""
        CREATE OR REPLACE TABLE squashed_name_freq AS
        SELECT name_squashed, COUNT(*)::BIGINT AS freq
        FROM s23
        WHERE name_squashed <> ''
        GROUP BY name_squashed
    """)

    # Address token ranking used by the composite keys.
    con.execute(f"""
        CREATE OR REPLACE TABLE addr_top6_ascii AS
        WITH base AS (
            SELECT s.s23_rid, u.token, f.freq
            FROM s23 s
            CROSS JOIN UNNEST(s.addr_tokens_ascii) AS u(token)
            JOIN addr_token_freq f
              ON f.kind = 'ascii' AND f.token = u.token
            WHERE u.token <> ''
              AND f.freq <= {MAX_ADDR_POSTING}
        ), ranked AS (
            SELECT
                s23_rid,
                token,
                freq,
                ROW_NUMBER() OVER (
                    PARTITION BY s23_rid
                    ORDER BY freq ASC, LENGTH(token) DESC, token ASC
                ) AS rn
            FROM base
        )
        SELECT s23_rid, token, freq, rn
        FROM ranked
        WHERE rn <= {MAX_ADDR_TOKENS_FOR_PAIRS}
    """)

    # Address number + token keys.
    con.execute(f"""
        CREATE OR REPLACE TABLE addr_num_keys AS
        SELECT DISTINCT
            s.s23_rid,
            CONCAT(n.num, '|', t.token) AS key
        FROM s23 s
        CROSS JOIN UNNEST(list_slice(s.numbers_native, 1, {MAX_ADDR_NUMS_FOR_PAIRS})) AS n(num)
        JOIN addr_top6_ascii t
          ON t.s23_rid = s.s23_rid
        WHERE n.num <> '' AND t.token <> ''
    """)

    con.execute("""
        CREATE OR REPLACE TABLE addr_num_freq AS
        SELECT key, COUNT(*)::BIGINT AS freq
        FROM addr_num_keys
        GROUP BY key
    """)

    # Address token-pair keys.
    con.execute("""
        CREATE OR REPLACE TABLE addr_pair_keys AS
        SELECT
            a.s23_rid,
            CONCAT(a.token, '|', b.token) AS key
        FROM addr_top6_ascii a
        JOIN addr_top6_ascii b
          ON a.s23_rid = b.s23_rid
         AND a.rn < b.rn
    """)
    con.execute("""
        CREATE OR REPLACE TABLE addr_pair_freq AS
        SELECT key, COUNT(*)::BIGINT AS freq
        FROM addr_pair_keys
        GROUP BY key
    """)

    # Filtered postings used by retrieval.
    con.execute(f"""
        CREATE OR REPLACE TABLE idx_exact_native_name AS
        SELECT s23_rid, name_native AS key
        FROM s23 s JOIN name_native_freq f USING(name_native)
        WHERE name_native <> '' AND f.freq <= {MAX_NAME_POSTING}
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE idx_exact_ascii_name AS
        SELECT s23_rid, name_ascii AS key
        FROM s23 s JOIN name_ascii_freq f USING(name_ascii)
        WHERE name_ascii <> '' AND f.freq <= {MAX_NAME_POSTING}
    """)

    con.execute(f"""
        CREATE OR REPLACE TABLE idx_rare_name_native AS
        SELECT nt.s23_rid, nt.token AS key
        FROM name_tokens nt
        JOIN name_token_freq f USING(kind, token)
        WHERE nt.kind='native' AND f.freq <= {MAX_NAME_POSTING}
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE idx_rare_name_ascii AS
        SELECT nt.s23_rid, nt.token AS key
        FROM name_tokens nt
        JOIN name_token_freq f USING(kind, token)
        WHERE nt.kind='ascii' AND f.freq <= {MAX_NAME_POSTING}
    """)

    con.execute(f"""
        CREATE OR REPLACE TABLE idx_rare_addr_native AS
        SELECT a.s23_rid, a.token AS key
        FROM addr_tokens a
        JOIN addr_token_freq f USING(kind, token)
        WHERE a.kind='native' AND f.freq <= {MAX_ADDR_POSTING}
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE idx_rare_addr_ascii AS
        SELECT a.s23_rid, a.token AS key
        FROM addr_tokens a
        JOIN addr_token_freq f USING(kind, token)
        WHERE a.kind='ascii' AND f.freq <= {MAX_ADDR_POSTING}
    """)

    con.execute(f"""
        CREATE OR REPLACE TABLE idx_postal AS
        SELECT s23_rid, postal AS key
        FROM s23
        CROSS JOIN UNNEST(postal_native) AS u(postal)
        WHERE postal <> ''
        QUALIFY COUNT(*) OVER (PARTITION BY postal) <= {MAX_ADDR_POSTING}
    """)

    con.execute(f"""
        CREATE OR REPLACE TABLE idx_prefix4 AS
        SELECT s23_rid, name_prefix4 AS key
        FROM s23 s JOIN prefix4_freq f USING(name_prefix4)
        WHERE name_prefix4 <> '' AND f.freq <= {MAX_PREFIX_POSTING}
    """)

    con.execute(f"""
        CREATE OR REPLACE TABLE idx_qgram AS
        SELECT q.s23_rid, q.qgram AS key
        FROM qgrams q JOIN qgram_freq f USING(qgram)
        WHERE f.freq <= {MAX_QGRAM_POSTING}
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE idx_qgram_100 AS
        SELECT q.s23_rid, q.qgram AS key
        FROM qgrams q JOIN qgram_freq f USING(qgram)
        WHERE f.freq <= {MAX_COMPOSITE_POSTING}
    """)

    con.execute(f"""
        CREATE OR REPLACE TABLE idx_squashed_exact AS
        SELECT s23_rid, name_squashed AS key
        FROM s23 s JOIN squashed_name_freq f USING(name_squashed)
        WHERE name_squashed <> '' AND f.freq <= {MAX_COMPOSITE_POSTING}
    """)

    con.execute(f"""
        CREATE OR REPLACE TABLE idx_addr_num AS
        SELECT k.s23_rid, k.key
        FROM addr_num_keys k JOIN addr_num_freq f USING(key)
        WHERE f.freq <= {MAX_COMPOSITE_POSTING}
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE idx_addr_pair AS
        SELECT k.s23_rid, k.key
        FROM addr_pair_keys k JOIN addr_pair_freq f USING(key)
        WHERE f.freq <= {MAX_COMPOSITE_POSTING}
    """)

    # Secondary indexes for frequent join keys. DuckDB may choose hash joins instead.
    index_specs = [
        ("idx_exact_native_name", "key"),
        ("idx_exact_ascii_name", "key"),
        ("idx_rare_name_native", "key"),
        ("idx_rare_name_ascii", "key"),
        ("idx_rare_addr_native", "key"),
        ("idx_rare_addr_ascii", "key"),
        ("idx_postal", "key"),
        ("idx_prefix4", "key"),
        ("idx_qgram", "key"),
        ("idx_qgram_100", "key"),
        ("idx_squashed_exact", "key"),
        ("idx_addr_num", "key"),
        ("idx_addr_pair", "key"),
    ]
    for table, column in index_specs:
        try:
            con.execute(f"CREATE INDEX IF NOT EXISTS ix_{table}_{column} ON {table}({column})")
        except Exception as e:
            print("Secondary index warning:", table, e)

    con.execute("ANALYZE")
    print(f"Index build completed in {time.time()-t0:,.1f}s")
    return con

con = build_full_index(DB_PATH, overwrite=OVERWRITE_DB)

In [ ]:
# ============================================================
# 9. INDEX DIAGNOSTICS
# ============================================================
for table in [
    "s1", "s23",
    "idx_exact_native_name", "idx_exact_ascii_name",
    "idx_rare_name_native", "idx_rare_name_ascii",
    "idx_rare_addr_native", "idx_rare_addr_ascii",
    "idx_postal", "idx_prefix4", "idx_qgram", "idx_qgram_100",
    "idx_squashed_exact", "idx_addr_num", "idx_addr_pair",
]:
    print(f"{table:28s}", f"{con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]:,}")

## Candidate-generation query

For each S1 batch, the query below:

- derives the rarest usable query-side name/address keys;
- retrieves all matching postings for those keys;
- includes exact name, postal, prefix and q-gram evidence;
- adds the targeted squashed-name and address-composite evidence;
- aggregates blocker evidence into `block_score`;
- ranks deterministically;
- keeps the top 150 candidates per S1.

The q-gram path intentionally has two layers: the normal q-gram weight and the additional low-frequency squashed-q-gram weight. This mirrors the targeted experiment, while avoiding a duplicate storage structure because the existing 3-gram representation is already computed from the squashed ASCII name.

In [ ]:
# ============================================================
# 10. BATCH CANDIDATE GENERATOR
# ============================================================
def generate_candidates_batch(con, s1_batch):
    if s1_batch.empty:
        return pd.DataFrame(columns=["s1_rid", "s23_rid", "block_score", "hit_count", "rank"])

    con.register("batch_s1_df", s1_batch)
    try:
        query = f"""
        WITH
        batch AS (
            SELECT * FROM batch_s1_df
        ),

        name_native_keys AS (
            SELECT
                b.s1_rid,
                u.token AS key,
                f.freq,
                ROW_NUMBER() OVER (
                    PARTITION BY b.s1_rid
                    ORDER BY f.freq ASC, LENGTH(u.token) DESC, u.token ASC
                ) AS rn
            FROM batch b
            CROSS JOIN UNNEST(b.name_tokens_native) AS u(token)
            JOIN name_token_freq f
              ON f.kind='native' AND f.token=u.token
            WHERE u.token <> ''
              AND f.freq <= {MAX_NAME_POSTING}
        ),
        name_native_top AS (
            SELECT s1_rid, key
            FROM name_native_keys
            WHERE rn <= {RARE_NAME_KEYS}
        ),

        name_ascii_keys AS (
            SELECT
                b.s1_rid,
                u.token AS key,
                f.freq,
                ROW_NUMBER() OVER (
                    PARTITION BY b.s1_rid
                    ORDER BY f.freq ASC, LENGTH(u.token) DESC, u.token ASC
                ) AS rn
            FROM batch b
            CROSS JOIN UNNEST(b.name_tokens_ascii) AS u(token)
            JOIN name_token_freq f
              ON f.kind='ascii' AND f.token=u.token
            WHERE u.token <> ''
              AND f.freq <= {MAX_NAME_POSTING}
        ),
        name_ascii_top AS (
            SELECT s1_rid, key
            FROM name_ascii_keys
            WHERE rn <= {RARE_NAME_KEYS}
        ),

        addr_native_keys AS (
            SELECT
                b.s1_rid,
                u.token AS key,
                f.freq,
                ROW_NUMBER() OVER (
                    PARTITION BY b.s1_rid
                    ORDER BY f.freq ASC, LENGTH(u.token) DESC, u.token ASC
                ) AS rn
            FROM batch b
            CROSS JOIN UNNEST(b.addr_tokens_native) AS u(token)
            JOIN addr_token_freq f
              ON f.kind='native' AND f.token=u.token
            WHERE u.token <> ''
              AND f.freq <= {MAX_ADDR_POSTING}
        ),
        addr_native_top AS (
            SELECT s1_rid, key
            FROM addr_native_keys
            WHERE rn <= {RARE_ADDR_KEYS}
        ),

        addr_ascii_keys AS (
            SELECT
                b.s1_rid,
                u.token AS key,
                f.freq,
                ROW_NUMBER() OVER (
                    PARTITION BY b.s1_rid
                    ORDER BY f.freq ASC, LENGTH(u.token) DESC, u.token ASC
                ) AS rn
            FROM batch b
            CROSS JOIN UNNEST(b.addr_tokens_ascii) AS u(token)
            JOIN addr_token_freq f
              ON f.kind='ascii' AND f.token=u.token
            WHERE u.token <> ''
              AND f.freq <= {MAX_ADDR_POSTING}
        ),
        addr_ascii_top AS (
            SELECT s1_rid, key
            FROM addr_ascii_keys
            WHERE rn <= {RARE_ADDR_KEYS}
        ),

        qgram_keys AS (
            SELECT
                b.s1_rid,
                u.qgram AS key,
                f.freq,
                ROW_NUMBER() OVER (
                    PARTITION BY b.s1_rid
                    ORDER BY f.freq ASC, u.qgram ASC
                ) AS rn
            FROM batch b
            CROSS JOIN UNNEST(b.name_qgrams) AS u(qgram)
            JOIN qgram_freq f ON f.qgram=u.qgram
            WHERE u.qgram <> ''
              AND f.freq <= {MAX_QGRAM_POSTING}
        ),
        qgram_top AS (
            SELECT s1_rid, key
            FROM qgram_keys
            WHERE rn <= {RARE_QGRAM_KEYS}
        ),

        qgram_100_keys AS (
            SELECT
                b.s1_rid,
                u.qgram AS key,
                f.freq,
                ROW_NUMBER() OVER (
                    PARTITION BY b.s1_rid
                    ORDER BY f.freq ASC, u.qgram ASC
                ) AS rn
            FROM batch b
            CROSS JOIN UNNEST(b.name_qgrams) AS u(qgram)
            JOIN qgram_freq f ON f.qgram=u.qgram
            WHERE u.qgram <> ''
              AND f.freq <= {MAX_COMPOSITE_POSTING}
        ),
        qgram_100_top AS (
            SELECT s1_rid, key
            FROM qgram_100_keys
            WHERE rn <= {RARE_QGRAM_KEYS}
        ),

        addr_top6 AS (
            SELECT
                b.s1_rid,
                u.token AS token,
                f.freq,
                ROW_NUMBER() OVER (
                    PARTITION BY b.s1_rid
                    ORDER BY f.freq ASC, LENGTH(u.token) DESC, u.token ASC
                ) AS rn
            FROM batch b
            CROSS JOIN UNNEST(b.addr_tokens_ascii) AS u(token)
            JOIN addr_token_freq f
              ON f.kind='ascii' AND f.token=u.token
            WHERE u.token <> ''
              AND f.freq <= {MAX_ADDR_POSTING}
        ),
        addr_top6_final AS (
            SELECT s1_rid, token, rn
            FROM addr_top6
            WHERE rn <= {MAX_ADDR_TOKENS_FOR_PAIRS}
        ),

        addr_num_query AS (
            SELECT DISTINCT
                b.s1_rid,
                CONCAT(n.num, '|', t.token) AS key
            FROM batch b
            CROSS JOIN UNNEST(list_slice(b.numbers_native, 1, {MAX_ADDR_NUMS_FOR_PAIRS})) AS n(num)
            JOIN addr_top6_final t
              ON t.s1_rid=b.s1_rid
            WHERE n.num <> '' AND t.token <> ''
        ),

        addr_pair_query AS (
            SELECT
                a.s1_rid,
                CONCAT(a.token, '|', b.token) AS key
            FROM addr_top6_final a
            JOIN addr_top6_final b
              ON a.s1_rid=b.s1_rid AND a.rn < b.rn
        ),

        evidence AS (
            -- Base blocker.
            SELECT b.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['exact_native_name']}::DOUBLE AS weight
            FROM batch b
            JOIN idx_exact_native_name i ON i.key=b.name_native
            WHERE b.name_native <> ''

            UNION ALL
            SELECT b.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['exact_ascii_name']}::DOUBLE
            FROM batch b
            JOIN idx_exact_ascii_name i ON i.key=b.name_ascii
            WHERE b.name_ascii <> ''

            UNION ALL
            SELECT k.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['rare_name_native']}::DOUBLE
            FROM name_native_top k
            JOIN idx_rare_name_native i ON i.key=k.key

            UNION ALL
            SELECT k.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['rare_name_ascii']}::DOUBLE
            FROM name_ascii_top k
            JOIN idx_rare_name_ascii i ON i.key=k.key

            UNION ALL
            SELECT k.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['rare_addr_native']}::DOUBLE
            FROM addr_native_top k
            JOIN idx_rare_addr_native i ON i.key=k.key

            UNION ALL
            SELECT k.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['rare_addr_ascii']}::DOUBLE
            FROM addr_ascii_top k
            JOIN idx_rare_addr_ascii i ON i.key=k.key

            UNION ALL
            SELECT b.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['postal']}::DOUBLE
            FROM batch b
            CROSS JOIN UNNEST(b.postal_native) AS u(postal)
            JOIN idx_postal i ON i.key=u.postal
            WHERE u.postal <> ''

            UNION ALL
            SELECT b.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['prefix4']}::DOUBLE
            FROM batch b
            JOIN idx_prefix4 i ON i.key=b.name_prefix4
            WHERE b.name_prefix4 <> ''

            UNION ALL
            SELECT k.s1_rid, i.s23_rid, {BLOCK_WEIGHTS['qgram']}::DOUBLE
            FROM qgram_top k
            JOIN idx_qgram i ON i.key=k.key

            -- Targeted additions.
            UNION ALL
            SELECT b.s1_rid, i.s23_rid, {TARGET_WEIGHTS['squashed_exact']}::DOUBLE
            FROM batch b
            JOIN idx_squashed_exact i ON i.key=b.name_squashed
            WHERE b.name_squashed <> ''

            UNION ALL
            SELECT k.s1_rid, i.s23_rid, {TARGET_WEIGHTS['squashed_qgram']}::DOUBLE
            FROM qgram_100_top k
            JOIN idx_qgram_100 i ON i.key=k.key

            UNION ALL
            SELECT q.s1_rid, i.s23_rid, {TARGET_WEIGHTS['addr_num']}::DOUBLE
            FROM addr_num_query q
            JOIN idx_addr_num i ON i.key=q.key

            UNION ALL
            SELECT q.s1_rid, i.s23_rid, {TARGET_WEIGHTS['addr_pair']}::DOUBLE
            FROM addr_pair_query q
            JOIN idx_addr_pair i ON i.key=q.key
        ),

        aggregated AS (
            SELECT
                s1_rid,
                s23_rid,
                SUM(weight)::DOUBLE AS block_score,
                COUNT(*)::BIGINT AS hit_count
            FROM evidence
            GROUP BY s1_rid, s23_rid
        ),

        ranked AS (
            SELECT
                s1_rid,
                s23_rid,
                block_score,
                hit_count,
                ROW_NUMBER() OVER (
                    PARTITION BY s1_rid
                    ORDER BY block_score DESC, hit_count DESC, s23_rid ASC
                ) AS rank
            FROM aggregated
        )

        SELECT s1_rid, s23_rid, block_score, hit_count, rank
        FROM ranked
        WHERE rank <= {EXPANSION_K}
        ORDER BY s1_rid, rank
        """
        out = con.execute(query).df()
    finally:
        con.unregister("batch_s1_df")

    return out

In [ ]:
# ============================================================
# 11. PAIR FEATURE GENERATION (35 FEATURES)
# ============================================================
def _pair_cosine(vec, s1_texts, cand_texts, s1_pos, cand_pos):
    s1_matrix = vec.transform(s1_texts)
    cand_matrix = vec.transform(cand_texts)

    # Pairwise only: never construct an all-by-all cosine matrix.
    out = np.empty(len(s1_pos), dtype=np.float32)
    for start in range(0, len(s1_pos), TFIDF_PAIR_BATCH):
        end = min(start + TFIDF_PAIR_BATCH, len(s1_pos))
        a = s1_matrix[s1_pos[start:end]]
        b = cand_matrix[cand_pos[start:end]]
        out[start:end] = np.asarray(a.multiply(b).sum(axis=1)).ravel().astype(np.float32)
    return out

def build_model_features(pair_df):
    if pair_df.empty:
        return pd.DataFrame(columns=FEATURE_COLUMNS)

    x = pair_df.copy()

    # --------------------------------------------------------
    # Continuous string similarity.
    # Native ratios are kept separately from ASCII ratios.
    # Token-level fuzzy functions use the ASCII normalized form.
    # --------------------------------------------------------
    x["name_ratio_native"] = ratio_series(
        x["name_native_s1"], x["name_native_cand"], fuzz.ratio
    )
    x["name_ratio_ascii"] = ratio_series(
        x["name_ascii_s1"], x["name_ascii_cand"], fuzz.ratio
    )
    x["name_token_sort"] = ratio_series(
        x["name_ascii_s1"], x["name_ascii_cand"], fuzz.token_sort_ratio
    )
    x["name_token_set"] = ratio_series(
        x["name_ascii_s1"], x["name_ascii_cand"], fuzz.token_set_ratio
    )
    x["name_jaccard_native"] = list_jaccard_series(
        x["name_tokens_native_s1"], x["name_tokens_native_cand"]
    )
    x["name_jaccard_ascii"] = list_jaccard_series(
        x["name_tokens_ascii_s1"], x["name_tokens_ascii_cand"]
    )
    x["name_shared_tokens"] = shared_count_series(
        x["name_tokens_ascii_s1"], x["name_tokens_ascii_cand"]
    )
    x["name_exact_native"] = (
        (x["name_native_s1"] != "") &
        (x["name_native_s1"] == x["name_native_cand"])
    ).astype(np.int8)
    x["name_exact_ascii"] = (
        (x["name_ascii_s1"] != "") &
        (x["name_ascii_s1"] == x["name_ascii_cand"])
    ).astype(np.int8)

    x["address_ratio_native"] = ratio_series(
        x["addr_native_s1"], x["addr_native_cand"], fuzz.ratio
    )
    x["address_ratio_ascii"] = ratio_series(
        x["addr_ascii_s1"], x["addr_ascii_cand"], fuzz.ratio
    )
    x["address_token_sort"] = ratio_series(
        x["addr_ascii_s1"], x["addr_ascii_cand"], fuzz.token_sort_ratio
    )
    x["address_token_set"] = ratio_series(
        x["addr_ascii_s1"], x["addr_ascii_cand"], fuzz.token_set_ratio
    )
    x["address_jaccard_native"] = list_jaccard_series(
        x["addr_tokens_native_s1"], x["addr_tokens_native_cand"]
    )
    x["address_jaccard_ascii"] = list_jaccard_series(
        x["addr_tokens_ascii_s1"], x["addr_tokens_ascii_cand"]
    )
    x["address_shared_tokens"] = shared_count_series(
        x["addr_tokens_ascii_s1"], x["addr_tokens_ascii_cand"]
    )
    x["address_exact_native"] = (
        (x["addr_native_s1"] != "") &
        (x["addr_native_s1"] == x["addr_native_cand"])
    ).astype(np.int8)
    x["address_exact_ascii"] = (
        (x["addr_ascii_s1"] != "") &
        (x["addr_ascii_s1"] == x["addr_ascii_cand"])
    ).astype(np.int8)

    # --------------------------------------------------------
    # Numeric / postal / country structure.
    # --------------------------------------------------------
    x["number_overlap"] = number_overlap_series(
        x["numbers_s1"], x["numbers_cand"]
    )
    x["number_shared_count"] = number_shared_series(
        x["numbers_s1"], x["numbers_cand"]
    )
    x["postal_match"] = np.fromiter(
        (
            int(bool(set(_to_list(a)) & set(_to_list(b))))
            for a, b in zip(x["postal_s1"], x["postal_cand"])
        ),
        dtype=np.int8,
        count=len(x),
    )
    x["country_match"] = (
        (x["country_s1"] != "") &
        (x["country_s1"] == x["country_cand"])
    ).astype(np.int8)

    # --------------------------------------------------------
    # Missingness + length structure.
    # --------------------------------------------------------
    x["name_missing"] = (x["name_ascii_s1"] == "").astype(np.int8)
    x["candidate_name_missing"] = (x["name_ascii_cand"] == "").astype(np.int8)
    x["address_missing"] = (x["addr_ascii_s1"] == "").astype(np.int8)
    x["candidate_address_missing"] = (x["addr_ascii_cand"] == "").astype(np.int8)

    x["name_len_ratio"] = len_ratio_series(
        x["name_ascii_s1"], x["name_ascii_cand"]
    )
    x["address_len_ratio"] = len_ratio_series(
        x["addr_ascii_s1"], x["addr_ascii_cand"]
    )

    x["name_addr_ratio_mean"] = (
        x["name_ratio_ascii"] + x["address_ratio_ascii"]
    ) / 2.0
    x["max_field_ratio"] = np.maximum(
        x["name_ratio_ascii"], x["address_ratio_ascii"]
    )

    # --------------------------------------------------------
    # Row-aligned sparse TF-IDF cosine features.
    # --------------------------------------------------------
    source_unique = x[["s1_rid", "name_ascii_s1", "addr_ascii_s1"]].drop_duplicates("s1_rid")
    cand_unique = x[["s23_rid", "name_ascii_cand", "addr_ascii_cand"]].drop_duplicates("s23_rid")

    source_pos = {int(rid): i for i, rid in enumerate(source_unique["s1_rid"].tolist())}
    cand_pos = {int(rid): i for i, rid in enumerate(cand_unique["s23_rid"].tolist())}

    s1_idx = np.fromiter((source_pos[int(v)] for v in x["s1_rid"]), dtype=np.int32, count=len(x))
    cand_idx = np.fromiter((cand_pos[int(v)] for v in x["s23_rid"]), dtype=np.int32, count=len(x))

    source_names = source_unique["name_ascii_s1"].tolist()
    cand_names = cand_unique["name_ascii_cand"].tolist()
    source_addrs = source_unique["addr_ascii_s1"].tolist()
    cand_addrs = cand_unique["addr_ascii_cand"].tolist()

    x["name_tfidf_char_cosine"] = _pair_cosine(
        NAME_CHAR_VECTORIZER, source_names, cand_names, s1_idx, cand_idx
    )
    x["name_tfidf_word_cosine"] = _pair_cosine(
        NAME_WORD_VECTORIZER, source_names, cand_names, s1_idx, cand_idx
    )
    x["address_tfidf_char_cosine"] = _pair_cosine(
        ADDRESS_CHAR_VECTORIZER, source_addrs, cand_addrs, s1_idx, cand_idx
    )
    x["address_tfidf_word_cosine"] = _pair_cosine(
        ADDRESS_WORD_VECTORIZER, source_addrs, cand_addrs, s1_idx, cand_idx
    )

    features = x[FEATURE_COLUMNS].astype(np.float32, copy=False)
    features = features.replace([np.inf, -np.inf], 0.0).fillna(0.0)
    return features

print("35-feature model feature function loaded.")

In [ ]:
# ============================================================
# 12. FETCH PAIR DATA FOR SCORING
# ============================================================
def materialize_pair_batch(con, candidate_df, s1_batch):
    if candidate_df.empty:
        return pd.DataFrame()

    candidate_ids = candidate_df[["s23_rid"]].drop_duplicates().copy()
    candidate_ids["s23_rid"] = candidate_ids["s23_rid"].astype(np.int64)

    con.register("batch_candidate_ids", candidate_ids)
    try:
        cand_df = con.execute("""
            SELECT
                s23_rid,
                entity_id,
                source,
                country,
                name_native,
                name_ascii,
                addr_native,
                addr_ascii,
                name_tokens_native,
                name_tokens_ascii,
                addr_tokens_native,
                addr_tokens_ascii,
                postal_native,
                numbers_native
            FROM s23
            JOIN batch_candidate_ids USING(s23_rid)
        """).df()
    finally:
        con.unregister("batch_candidate_ids")

    s1_cols = [
        "s1_rid", "country", "name_native", "name_ascii", "addr_native", "addr_ascii",
        "name_tokens_native", "name_tokens_ascii", "addr_tokens_native", "addr_tokens_ascii",
        "postal_native", "numbers_native"
    ]

    s1_small = s1_batch[s1_cols].copy().rename(columns={
        "country": "country_s1",
        "name_native": "name_native_s1",
        "name_ascii": "name_ascii_s1",
        "addr_native": "addr_native_s1",
        "addr_ascii": "addr_ascii_s1",
        "name_tokens_native": "name_tokens_native_s1",
        "name_tokens_ascii": "name_tokens_ascii_s1",
        "addr_tokens_native": "addr_tokens_native_s1",
        "addr_tokens_ascii": "addr_tokens_ascii_s1",
        "postal_native": "postal_s1",
        "numbers_native": "numbers_s1",
    })

    cand_df = cand_df.rename(columns={
        "country": "country_cand",
        "name_native": "name_native_cand",
        "name_ascii": "name_ascii_cand",
        "addr_native": "addr_native_cand",
        "addr_ascii": "addr_ascii_cand",
        "name_tokens_native": "name_tokens_native_cand",
        "name_tokens_ascii": "name_tokens_ascii_cand",
        "addr_tokens_native": "addr_tokens_native_cand",
        "addr_tokens_ascii": "addr_tokens_ascii_cand",
        "postal_native": "postal_cand",
        "numbers_native": "numbers_cand",
        "entity_id": "candidate_entity_id",
        "source": "candidate_source",
    })

    x = candidate_df.merge(s1_small, on="s1_rid", how="left")
    x = x.merge(cand_df, on="s23_rid", how="left")

    required = [
        "s1_rid", "s23_rid", "block_score", "rank",
        "country_s1", "name_native_s1", "name_ascii_s1", "addr_native_s1", "addr_ascii_s1",
        "name_tokens_native_s1", "name_tokens_ascii_s1", "addr_tokens_native_s1", "addr_tokens_ascii_s1",
        "postal_s1", "numbers_s1",
        "country_cand", "name_native_cand", "name_ascii_cand", "addr_native_cand", "addr_ascii_cand",
        "name_tokens_native_cand", "name_tokens_ascii_cand", "addr_tokens_native_cand", "addr_tokens_ascii_cand",
        "postal_cand", "numbers_cand",
    ]
    missing = [c for c in required if c not in x.columns]
    if missing:
        raise KeyError(f"Pair materialization missing columns: {missing}. Columns: {list(x.columns)}")

    return x

In [ ]:
# ============================================================
# 13. FAST FULL TEST INFERENCE — TOP 3 / MEMORY SAFE
# ============================================================
import gc

MATCH_PARTS_DIR = RUN_DIR / "matching_parts"
CAND_PARTS_DIR = RUN_DIR / "candidate_parts"
STATS_DIR = RUN_DIR / "stats"

for p in [MATCH_PARTS_DIR, CAND_PARTS_DIR, STATS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def score_candidate_subset(con, candidate_df, s1_batch):
    if candidate_df.empty:
        return pd.DataFrame(
            columns=["s1_rid", "s23_rid", "rank", "block_score", "match_score"]
        )

    pair_df = materialize_pair_batch(con, candidate_df, s1_batch)
    features = build_model_features(pair_df)
    probs = MODEL.predict_proba(features)[:, 1].astype(np.float32)

    out = candidate_df[
        ["s1_rid", "s23_rid", "rank", "block_score", "hit_count"]
    ].copy()
    out["match_score"] = probs
    return out

def entity_output_rows(con, scored_df, candidate_df, s1_batch):
    cand_map = {}

    if not candidate_df.empty:
        output_ids = candidate_df[["s23_rid"]].drop_duplicates().copy()
        output_ids["s23_rid"] = output_ids["s23_rid"].astype(np.int64)

        con.register("output_candidate_ids", output_ids)
        try:
            rows = con.execute("""
                SELECT s23_rid, entity_id
                FROM s23
                JOIN output_candidate_ids USING(s23_rid)
            """).fetchall()
        finally:
            con.unregister("output_candidate_ids")

        cand_map = {int(rid): str(eid) for rid, eid in rows}

    candidate_groups = (
        {
            int(k): g.sort_values("rank")
            for k, g in candidate_df.groupby("s1_rid", sort=True)
        }
        if not candidate_df.empty else {}
    )

    score_groups = (
        {
            int(k): g.sort_values("rank")
            for k, g in scored_df.groupby("s1_rid", sort=True)
        }
        if not scored_df.empty else {}
    )

    match_rows = []
    cand_rows = []

    for rid, eid in zip(s1_batch["s1_rid"], s1_batch["entity_id"]):
        rid = int(rid)
        eid = str(eid)

        cg = candidate_groups.get(rid)
        sg = score_groups.get(rid)

        candidate_ids = (
            [] if cg is None else [
                cand_map[int(v)]
                for v in cg["s23_rid"].tolist()
                if int(v) in cand_map
            ]
        )

        matched_ids = (
            [] if sg is None else [
                cand_map[int(v)]
                for v in sg.loc[
                    sg["match_score"] >= DECISION_THRESHOLD,
                    "s23_rid",
                ].tolist()
                if int(v) in cand_map
            ]
        )

        candidate_ids = list(dict.fromkeys(candidate_ids))
        matched_ids = list(dict.fromkeys(matched_ids))

        cand_rows.append({
            "source1_entity_id": eid,
            "candidate_entity_ids": ",".join(candidate_ids),
        })
        match_rows.append({
            "source1_entity_id": eid,
            "matched_entity_ids": ",".join(matched_ids),
        })

    return pd.DataFrame(match_rows), pd.DataFrame(cand_rows)

def save_part(match_df, cand_df, batch_id):
    match_path = MATCH_PARTS_DIR / f"part-{batch_id:05d}.tsv"
    cand_path = CAND_PARTS_DIR / f"part-{batch_id:05d}.tsv"

    match_df[["source1_entity_id", "matched_entity_ids"]].to_csv(
        match_path, sep="\t", index=False
    )
    cand_df[["source1_entity_id", "candidate_entity_ids"]].to_csv(
        cand_path, sep="\t", index=False
    )

TOTAL_TEST_S1 = int(
    con.execute("SELECT COUNT(*) FROM s1").fetchone()[0]
)

PROCESS_TEST_S1 = (
    min(TOTAL_TEST_S1, SMOKE_TEST_S1)
    if SMOKE_TEST
    else TOTAL_TEST_S1
)

n_batches = math.ceil(PROCESS_TEST_S1 / BLOCK_BATCH_S1)
run_start = time.time()
batch_stats = []

print(
    f"PROCESSING {PROCESS_TEST_S1:,} S1 rows with TOP-{EXPANSION_K} candidates only."
)
print(f"Batch size: {BLOCK_BATCH_S1:,} | batches: {n_batches:,}")

for batch_id, start in enumerate(
    range(0, PROCESS_TEST_S1, BLOCK_BATCH_S1)
):
    end = min(start + BLOCK_BATCH_S1, PROCESS_TEST_S1)

    match_path = MATCH_PARTS_DIR / f"part-{batch_id:05d}.tsv"
    cand_path = CAND_PARTS_DIR / f"part-{batch_id:05d}.tsv"
    stat_path = STATS_DIR / f"part-{batch_id:05d}.json"

    if RESUME and match_path.exists() and cand_path.exists() and stat_path.exists():
        batch_stats.append(
            json.loads(stat_path.read_text(encoding="utf-8"))
        )
        print(f"[{batch_id + 1}/{n_batches}] already complete")
        continue

    bt = time.time()

    # Load ONLY this S1 batch from DuckDB.
    s1_batch = con.execute(f"""
        SELECT *
        FROM s1
        WHERE s1_rid >= {int(start)}
          AND s1_rid < {int(end)}
        ORDER BY s1_rid
    """).df()

    # Candidate generator returns only top EXPANSION_K because the
    # global configuration is set to K=3.
    candidates = generate_candidates_batch(con, s1_batch)

    if not candidates.empty:
        candidates["s1_rid"] = candidates["s1_rid"].astype(np.int64)
        candidates["s23_rid"] = candidates["s23_rid"].astype(np.int64)
        candidates["rank"] = candidates["rank"].astype(np.int16)
        candidates.sort_values(["s1_rid", "rank"], inplace=True)
        candidates.reset_index(drop=True, inplace=True)

    # Score exactly the candidates written to candidate_pairs.tsv.
    scored = score_candidate_subset(
        con,
        candidates,
        s1_batch,
    )

    if not scored.empty:
        scored.sort_values(["s1_rid", "rank"], inplace=True)
        scored.reset_index(drop=True, inplace=True)

    match_df, cand_df = entity_output_rows(
        con,
        scored,
        candidates,
        s1_batch,
    )

    save_part(match_df, cand_df, batch_id)

    stats = {
        "batch_id": int(batch_id),
        "s1_start": int(start),
        "s1_end": int(end),
        "s1_rows": int(end - start),
        "candidate_rows": int(len(candidates)),
        "predicted_match_pairs": int(
            (scored["match_score"] >= DECISION_THRESHOLD).sum()
        ) if not scored.empty else 0,
        "avg_candidates_per_s1": float(
            len(candidates) / (end - start)
        ) if end > start else 0.0,
        "batch_seconds": float(time.time() - bt),
    }

    stat_path.write_text(
        json.dumps(stats, indent=2),
        encoding="utf-8",
    )
    batch_stats.append(stats)

    elapsed = time.time() - run_start
    rate = (end - start) / max(stats["batch_seconds"], 1e-9)
    remaining = max(PROCESS_TEST_S1 - end, 0)
    eta_min = remaining / max(rate, 1e-9) / 60.0

    print(
        f"[{batch_id + 1}/{n_batches}] "
        f"S1={start:,}:{end:,} | "
        f"cands={len(candidates):,} | "
        f"matches={stats['predicted_match_pairs']:,} | "
        f"batch={stats['batch_seconds']:.1f}s | "
        f"elapsed={elapsed / 60:.1f}m | "
        f"ETA={eta_min:.1f}m"
    )

    del s1_batch, candidates, scored, match_df, cand_df
    gc.collect()

print("Inference loop complete.")


In [ ]:
# ============================================================
# 14. CONCATENATE FINAL SUBMISSION FILES
# ============================================================
FINAL_MATCHING = OUTPUT_DIR / "matching_results.tsv"
FINAL_CANDIDATES = OUTPUT_DIR / "candidate_pairs.tsv"


def concat_tsv_parts(part_dir, output_path, columns):
    parts = sorted(Path(part_dir).glob("part-*.tsv"))
    if not parts:
        raise FileNotFoundError(f"No parts found in {part_dir}")

    with open(output_path, "w", encoding="utf-8", newline="") as out:
        out.write("\t".join(columns) + "\n")
        for part in parts:
            with open(part, "r", encoding="utf-8") as src:
                next(src, None)  # header
                for line in src:
                    out.write(line)
    return len(parts)

n_match_parts = concat_tsv_parts(
    MATCH_PARTS_DIR,
    FINAL_MATCHING,
    ["source1_entity_id", "matched_entity_ids"],
)

n_cand_parts = concat_tsv_parts(
    CAND_PARTS_DIR,
    FINAL_CANDIDATES,
    ["source1_entity_id", "candidate_entity_ids"],
)

print("matching_results.tsv:", FINAL_MATCHING)
print("candidate_pairs.tsv :", FINAL_CANDIDATES)
print("Matching parts      :", n_match_parts)
print("Candidate parts     :", n_cand_parts)

In [ ]:
# ============================================================
# 15. STREAMING SUBMISSION VALIDATION
# ============================================================
def iter_tsv_rows(path, chunksize=100_000):
    return pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
    )

def validate_submission(match_path, cand_path, expected_s1_ids_path=None):
    expected_rows = PROCESS_TEST_S1

    match_lines = 0
    cand_lines = 0
    duplicate_match_ids = 0
    duplicate_candidate_ids = 0
    subset_violations = 0
    self_match_violations = 0
    sequence_mismatches = 0

    match_iter = iter_tsv_rows(match_path)
    cand_iter = iter_tsv_rows(cand_path)

    total_candidate_count = 0
    total_match_count = 0

    for mdf, cdf in zip(match_iter, cand_iter):
        if len(mdf) != len(cdf):
            raise AssertionError("matching/candidate chunk lengths differ")

        for mid, mval, cid, cval in zip(
            mdf["source1_entity_id"],
            mdf["matched_entity_ids"],
            cdf["source1_entity_id"],
            cdf["candidate_entity_ids"],
        ):
            match_lines += 1
            cand_lines += 1

            if str(mid) != str(cid):
                sequence_mismatches += 1

            candidates = [x.strip() for x in str(cval).split(",") if x.strip()]
            matches = [x.strip() for x in str(mval).split(",") if x.strip()]

            total_candidate_count += len(candidates)
            total_match_count += len(matches)

            if len(candidates) != len(set(candidates)):
                duplicate_candidate_ids += 1
            if len(matches) != len(set(matches)):
                duplicate_match_ids += 1

            candidate_set = set(candidates)
            match_set = set(matches)

            if not match_set.issubset(candidate_set):
                subset_violations += 1

            if str(mid) in match_set:
                self_match_violations += 1

            # Every non-empty candidate/match ID must point to S2/S3.
            for x in candidates:
                if not (x.startswith("S2-") or x.startswith("S3-")):
                    raise AssertionError(f"Invalid candidate ID: {x}")
            for x in matches:
                if not (x.startswith("S2-") or x.startswith("S3-")):
                    raise AssertionError(f"Invalid matched ID: {x}")

    if match_lines != expected_rows or cand_lines != expected_rows:
        raise AssertionError(
            f"Expected {expected_rows:,} rows in each file; "
            f"got matching={match_lines:,}, candidates={cand_lines:,}"
        )

    if sequence_mismatches:
        raise AssertionError(f"S1 row-order/id mismatches: {sequence_mismatches}")
    if duplicate_match_ids:
        raise AssertionError(f"Rows with duplicate matched IDs: {duplicate_match_ids}")
    if duplicate_candidate_ids:
        raise AssertionError(f"Rows with duplicate candidate IDs: {duplicate_candidate_ids}")
    if subset_violations:
        raise AssertionError(f"Matched IDs not contained in candidate set: {subset_violations}")
    if self_match_violations:
        raise AssertionError(f"Self-match violations: {self_match_violations}")

    return {
        "expected_s1_rows": expected_rows,
        "matching_rows": match_lines,
        "candidate_rows": cand_lines,
        "total_candidate_links": total_candidate_count,
        "total_predicted_match_links": total_match_count,
        "average_candidates_per_s1": total_candidate_count / expected_rows,
        "average_predicted_matches_per_s1": total_match_count / expected_rows,
        "duplicate_candidate_rows": 0,
        "duplicate_match_rows": 0,
        "matched_not_in_candidate_rows": 0,
        "self_match_rows": 0,
    }

VALIDATION_SUMMARY = validate_submission(FINAL_MATCHING, FINAL_CANDIDATES)
print(json.dumps(VALIDATION_SUMMARY, indent=2))

In [ ]:
# ============================================================
# 16. OPTIONAL HEADER / SAMPLE INSPECTION
# ============================================================
print("\nmatching_results.tsv sample:")
print(pd.read_csv(FINAL_MATCHING, sep="\t", dtype=str, keep_default_na=False, nrows=10).to_string(index=False))

print("\ncandidate_pairs.tsv sample:")
print(pd.read_csv(FINAL_CANDIDATES, sep="\t", dtype=str, keep_default_na=False, nrows=10).to_string(index=False))

In [ ]:
# ============================================================
# 17. ZIP THE FINAL SUBMISSION
# ============================================================
ZIP_PATH = SUBMISSION_ROOT / ("amazon_ml_final_submission_smoke.zip" if SMOKE_TEST else "amazon_ml_final_submission.zip")
SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(FINAL_MATCHING, arcname="output/matching_results.tsv")
    zf.write(FINAL_CANDIDATES, arcname="output/candidate_pairs.tsv")

print("Created:", ZIP_PATH)
print("Size MB :", round(ZIP_PATH.stat().st_size / 1024**2, 2))

In [ ]:
# ============================================================
# 18. SAVE RUN METADATA
# ============================================================
metadata = {
    "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "smoke_test": SMOKE_TEST,
    "s1_rows_processed": int(PROCESS_TEST_S1),
    "test_s2_rows": int(s2_rows),
    "test_s3_rows": int(s3_rows),
    "feature_count": int(len(FEATURE_COLUMNS)),
    "decision_threshold": float(DECISION_THRESHOLD),
    "base_k": int(BASE_K),
    "expansion_k": int(EXPANSION_K),
    "expansion_rule": "FAST TOP-3; no expansion",
    "duckdb_memory": DUCKDB_MEMORY,
    "duckdb_threads": DUCKDB_THREADS,
    "saved_model": str(ARTIFACT_DIR / "checkpoint_model_8k.cbm"),
    "saved_model_note": SAVED_CONFIG.get("note"),
    "validation": VALIDATION_SUMMARY,
    "output_files": {
        "matching_results": str(FINAL_MATCHING),
        "candidate_pairs": str(FINAL_CANDIDATES),
        "zip": str(ZIP_PATH),
    },
}

meta_path = SUBMISSION_ROOT / "run_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

stats_df = pd.DataFrame(batch_stats)
stats_path = SUBMISSION_ROOT / "batch_stats.tsv"
stats_df.to_csv(stats_path, sep="\t", index=False)

print("Run metadata:", meta_path)
print("Batch stats  :", stats_path)

## Completion criteria

The run is complete when validation confirms:

```text
matching_rows == processed S1 rows
candidate_rows == processed S1 rows
matched IDs are a subset of candidate IDs
no duplicate IDs per row
no self-matches
```

The ZIP file is then written to Google Drive.
